# Text Generation (Decoding) with GPT-2

This notebook explores how language models generate text token-by-token, covering the full
pipeline from autoregressive decoding to practical sampling strategies.

**What we cover:**

1. **Autoregressive generation** -- the core loop of predict-next-token, append, repeat
2. **Greedy decoding** -- always picking the most likely token (and why it fails)
3. **Temperature scaling** -- controlling the sharpness of the output distribution
4. **Top-k sampling** -- restricting the candidate pool to the k most likely tokens
5. **Top-p (nucleus) sampling** -- dynamically sizing the candidate pool by cumulative probability
6. **KV caching** -- avoiding redundant computation during generation
7. **Visualizations and ablations** -- comparing strategies side by side

All code is self-contained. We build a tiny GPT-2 from scratch, train it on toy data,
then use it to illustrate every decoding technique.

In [ ]:
%matplotlib inline

import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Model Architecture

We use intentionally tiny dimensions so every cell runs in seconds, not minutes.
The architecture is faithful to GPT-2 -- only the scale differs.

We re-implement the full GPT-2 inline: causal multi-head attention,
a feed-forward block with GELU activation, and the transformer stack with learned
positional embeddings.

In [ ]:
VOCAB_SIZE = 256
EMBED_DIM = 64
N_HEADS = 4
N_LAYERS = 2
MAX_SEQ_LEN = 128


class MultiHeadAttention(nn.Module):
    """Causal multi-head self-attention."""

    def __init__(self, embed_dim, n_heads, max_seq_len):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(max_seq_len, max_seq_len)).unsqueeze(0).unsqueeze(0),
        )

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        scale = self.head_dim ** 0.5
        attn = (q @ k.transpose(-2, -1)) / scale
        attn = attn.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class TransformerBlock(nn.Module):
    """Pre-norm transformer block (GPT-2 style)."""

    def __init__(self, embed_dim, n_heads, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_heads, max_seq_len)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class GPT2(nn.Module):
    """Minimal GPT-2 language model."""

    def __init__(self, vocab_size, embed_dim, n_heads, n_layers, max_seq_len):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(max_seq_len, embed_dim)
        self.blocks = nn.ModuleList(
            [TransformerBlock(embed_dim, n_heads, max_seq_len) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(positions)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)


model = GPT2(VOCAB_SIZE, EMBED_DIM, N_HEADS, N_LAYERS, MAX_SEQ_LEN).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"vocab_size={VOCAB_SIZE}, embed_dim={EMBED_DIM}, n_heads={N_HEADS}, "
      f"n_layers={N_LAYERS}, max_seq_len={MAX_SEQ_LEN}")
print(f"Head dim: {EMBED_DIM // N_HEADS}")
print(f"Model parameters: {n_params:,}")

## 2. Toy Training Data and Quick Training

We create a small corpus of repetitive text so the model can learn predictable patterns
quickly. Since our vocab is byte-level (0--255), we encode raw ASCII characters directly.
We train for 100 steps to get the model past random outputs.

In [ ]:
corpus = (
    "the cat sat on the mat. "
    "the dog sat on the log. "
    "the cat and the dog sat on the mat. "
    "a cat sat on a mat. "
    "the bird flew over the mat. "
    "the cat chased the bird. "
) * 20

data = torch.tensor([ord(c) for c in corpus], dtype=torch.long, device=device)
print(f"Corpus length: {len(corpus)} characters / tokens")
print(f"Unique tokens used: {len(set(data.tolist()))}")
print(f"Sample: {corpus[:60]}")


def get_batch(data, batch_size=8, seq_len=64):
    """Sample random contiguous chunks from the data."""
    ix = torch.randint(0, len(data) - seq_len - 1, (batch_size,))
    x = torch.stack([data[i : i + seq_len] for i in ix])
    y = torch.stack([data[i + 1 : i + seq_len + 1] for i in ix])
    return x, y

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
losses = []

t0 = time.time()
model.train()
for step in range(100):
    xb, yb = get_batch(data, batch_size=16, seq_len=64)
    logits = model(xb)
    loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yb.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    if step % 20 == 0:
        print(f"Step {step:3d}  loss={loss.item():.3f}")

elapsed = time.time() - t0
print(f"\nTraining done in {elapsed:.1f}s  |  Final loss: {losses[-1]:.3f}")

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(losses, linewidth=1.2)
plt.xlabel("Training step")
plt.ylabel("Cross-entropy loss")
plt.title("Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The loss drops quickly because our corpus is small and repetitive. This is intentional --
we want the model to have learned *something* so the generation demonstrations are
illustrative rather than pure noise.

---
## 3. Autoregressive Generation Loop

The fundamental idea: given a prompt, feed it through the model to get logits for the
next token. Pick a token from those logits, append it to the sequence, and repeat.

```
prompt: [t1, t2, t3]
  -> model -> logits for t4 -> sample t4
  -> [t1, t2, t3, t4] -> model -> logits for t5 -> sample t5
  -> ...
```

The `decode` helper converts token IDs (bytes) back to readable text.

In [ ]:
def encode(text):
    """Encode a string to a list of byte-level token IDs."""
    return [ord(c) for c in text]


def decode(tokens):
    """Decode a list of byte-level token IDs back to a string."""
    return "".join(chr(t) for t in tokens if 0 <= t < 256)


@torch.no_grad()
def generate(model, prompt_tokens, max_new_tokens=50, temperature=1.0,
             top_k=None, top_p=None):
    """
    Autoregressive text generation with configurable decoding strategy.

    Args:
        model: the GPT-2 model
        prompt_tokens: list of int token IDs for the prompt
        max_new_tokens: how many tokens to generate
        temperature: softmax temperature (0 = greedy via argmax)
        top_k: if set, keep only top-k logits
        top_p: if set, apply nucleus sampling

    Returns:
        list of int token IDs (prompt + generated)
    """
    model.eval()
    tokens = list(prompt_tokens)

    for _ in range(max_new_tokens):
        context = tokens[-MAX_SEQ_LEN:]
        x = torch.tensor([context], dtype=torch.long, device=device)
        logits = model(x)[:, -1, :]  # logits for the last position (1, vocab)

        # --- Greedy ---
        if temperature == 0:
            next_token = logits.argmax(dim=-1).item()
            tokens.append(next_token)
            continue

        # --- Temperature ---
        logits = logits / temperature

        # --- Top-k filtering ---
        if top_k is not None:
            top_vals, _ = torch.topk(logits, top_k)
            threshold = top_vals[:, -1].unsqueeze(-1)
            logits = logits.masked_fill(logits < threshold, float("-inf"))

        # --- Top-p (nucleus) filtering ---
        if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_mask = cumulative_probs - F.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[sorted_mask] = float("-inf")
            logits = sorted_logits.scatter(1, sorted_indices, sorted_logits)

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1).item()
        tokens.append(next_token)

    return tokens


# Sanity check encode/decode round-trip
test_str = "the cat"
assert decode(encode(test_str)) == test_str
print(f"'{test_str}' -> {encode(test_str)} -> '{decode(encode(test_str))}'")
print("generate() function defined.")

---
## 4. Greedy Decoding

The simplest strategy: at each step pick the token with the highest probability
(argmax). This is deterministic -- the same prompt always produces the same output.

**Downside:** greedy decoding tends to get stuck in repetitive loops because it always
commits to the single most likely continuation.

In [ ]:
prompt = "the cat"
prompt_tokens = encode(prompt)

greedy_tokens = generate(model, prompt_tokens, max_new_tokens=80, temperature=0)
greedy_text = decode(greedy_tokens)

print("GREEDY DECODING")
print("=" * 60)
print(f"Prompt:    '{prompt}'")
print(f"Generated: '{greedy_text}'")

# Try a few different prompts
print("\nGREEDY DECODING -- MULTIPLE PROMPTS")
print("=" * 60)
for p in ["the ", "a cat", "the dog", "the bird"]:
    tokens = generate(model, encode(p), max_new_tokens=60, temperature=0)
    print(f"  '{p}' -> '{decode(tokens)}'")
    print()

Notice how greedy decoding often produces repetitive text -- the model finds a
high-probability loop and gets stuck in it. This is a well-known failure mode and
the main motivation for introducing randomness via sampling.

---
## 5. Temperature Scaling

Before sampling, we divide the logits by a temperature parameter $T$:

$$\text{logits}_{\text{scaled}} = \frac{\text{logits}}{T}$$

- **T < 1** makes the distribution *peakier* (more confident, less random)
- **T = 1** leaves the distribution unchanged
- **T > 1** makes the distribution *flatter* (more uniform, more random)
- **T -> 0** converges to greedy decoding

In [ ]:
# Visualize temperature effect on a single set of logits
model.eval()
with torch.no_grad():
    sample_input = torch.tensor([encode("the cat sat on the ")], device=device)
    raw_logits = model(sample_input)[:, -1, :].squeeze()  # (vocab_size,)

temperatures = [0.1, 0.5, 1.0, 2.0]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=False)
for ax, T in zip(axes, temperatures):
    scaled = raw_logits / T
    probs = F.softmax(scaled, dim=-1).cpu().numpy()
    top_idx = np.argsort(probs)[-20:][::-1]
    top_probs = probs[top_idx]
    labels = [chr(i) if 32 <= i < 127 else f"#{i}" for i in top_idx]

    ax.bar(range(len(top_probs)), top_probs, color="steelblue", alpha=0.8)
    ax.set_xticks(range(len(top_probs)))
    ax.set_xticklabels(labels, rotation=60, fontsize=7)
    ax.set_title(f"T = {T}")
    ax.set_ylabel("Probability" if ax == axes[0] else "")

fig.suptitle("Next-token probability distribution at different temperatures", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Generate text at various temperatures
prompt = "the cat"
prompt_tokens = encode(prompt)

print("TEMPERATURE COMPARISON")
print("=" * 60)
for T in [0.1, 0.5, 1.0, 1.5, 2.0]:
    tokens = generate(model, prompt_tokens, max_new_tokens=60, temperature=T)
    print(f"  T={T:<4} -> '{decode(tokens)}'")
    print()

**Observations:**
- At low temperature (0.1) the output closely resembles greedy decoding -- nearly deterministic.
- At T=1.0 we get the model's "natural" randomness.
- At high temperature (2.0) the text becomes increasingly incoherent as unlikely tokens
  get sampled more often.

Temperature alone does not prevent sampling from the long tail of very unlikely tokens.
That is where top-k and top-p come in.

---
## 6. Top-k Sampling

**Idea:** before sampling, zero out everything except the k most probable tokens.
This prevents the model from ever picking wildly unlikely tokens.

1. Compute logits
2. Find the k-th largest logit value
3. Set all logits below that threshold to $-\infty$
4. Apply softmax and sample

In [ ]:
# Visualize top-k cutoff on sorted probabilities
with torch.no_grad():
    probs_raw = F.softmax(raw_logits, dim=-1).cpu().numpy()

sorted_probs = np.sort(probs_raw)[::-1]
k_values = [5, 10, 20, 50]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sorted_probs[:80], linewidth=1.5, color="steelblue", label="Sorted probabilities")

colors = ["red", "orange", "green", "purple"]
for k, c in zip(k_values, colors):
    ax.axvline(x=k - 1, color=c, linestyle="--", linewidth=1.2, label=f"k={k} cutoff")

ax.set_xlabel("Rank (sorted by probability)")
ax.set_ylabel("Probability")
ax.set_title("Top-k Sampling: Sorted Token Probabilities with Cutoff Lines")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Generate with different top-k values
prompt = "the cat"
prompt_tokens = encode(prompt)

print("TOP-K SAMPLING (temperature=1.0)")
print("=" * 60)
for k in [3, 5, 10, 20, 50]:
    tokens = generate(model, prompt_tokens, max_new_tokens=60, temperature=1.0, top_k=k)
    print(f"  k={k:<3} -> '{decode(tokens)}'")
    print()

**Trade-off:** small k means safer but potentially repetitive text; large k allows
more diversity but risks incoherence. A common default in practice is k=50.

---
## 7. Top-p (Nucleus) Sampling

Top-k has a fixed candidate set size regardless of the distribution shape. Top-p
(nucleus sampling) adapts: it includes the *smallest* set of tokens whose cumulative
probability exceeds a threshold $p$.

Algorithm:
1. Sort tokens by probability (descending)
2. Compute cumulative sum
3. Mask out all tokens after the cumulative sum first exceeds $p$
4. Re-normalize and sample

When the model is confident (one token dominates), the nucleus is small.
When the model is uncertain, the nucleus grows automatically.

In [ ]:
# Visualize cumulative probability and nucleus cutoffs
cumulative = np.cumsum(sorted_probs)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(cumulative[:80], linewidth=1.5, color="steelblue", label="Cumulative probability")

p_values = [0.5, 0.8, 0.9, 0.95]
colors = ["red", "orange", "green", "purple"]
for p_val, c in zip(p_values, colors):
    cutoff_idx = np.searchsorted(cumulative, p_val)
    ax.axhline(y=p_val, color=c, linestyle="--", linewidth=1, alpha=0.7)
    ax.axvline(x=cutoff_idx, color=c, linestyle=":", linewidth=1, alpha=0.7)
    ax.annotate(f"p={p_val} (n={cutoff_idx+1})", xy=(cutoff_idx + 1, p_val - 0.03),
                fontsize=8, color=c)

ax.set_xlabel("Number of tokens included (sorted by probability)")
ax.set_ylabel("Cumulative probability")
ax.set_title("Nucleus Sampling: Cumulative Probability Cutoffs")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Generate with different top-p values
prompt = "the cat"
prompt_tokens = encode(prompt)

print("TOP-P (NUCLEUS) SAMPLING (temperature=1.0)")
print("=" * 60)
for p in [0.5, 0.8, 0.9, 0.95, 1.0]:
    tokens = generate(model, prompt_tokens, max_new_tokens=60, temperature=1.0, top_p=p)
    print(f"  p={p:<5} -> '{decode(tokens)}'")
    print()

**Why nucleus sampling is popular:** it adapts to the model's confidence at each step.
When the model "knows" what comes next (e.g., finishing a common word), the nucleus is
tiny and generation is nearly deterministic. When the model is uncertain (e.g., starting
a new sentence), the nucleus expands and allows creative exploration.

---
## 8. KV Cache for Efficient Generation

### The Problem

In the naive generation loop above, we feed the **entire** sequence through the model
at every step. If the sequence has length $L$, the attention computation is $O(L^2)$
per step, and generating $N$ tokens costs $O(N \cdot L^2)$ total.

But most of that work is wasted: the Key and Value projections for all previous tokens
are identical to what we computed in the previous step.

### The Solution: KV Cache

Cache the K and V tensors from each layer. At each new step, only compute Q, K, V for
the **new** token, concatenate K and V with the cache, and compute attention. This
reduces each step to $O(L)$ instead of $O(L^2)$.

In [ ]:
class MultiHeadAttentionWithCache(nn.Module):
    """Causal multi-head self-attention with optional KV cache."""

    def __init__(self, embed_dim, n_heads, max_seq_len):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(max_seq_len, max_seq_len)).unsqueeze(0).unsqueeze(0),
        )

    def forward(self, x, cache=None):
        B, T, C = x.shape
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # --- KV Cache logic ---
        if cache is not None:
            prev_k, prev_v = cache
            k = torch.cat([prev_k, k], dim=-2)  # concat along sequence dim
            v = torch.cat([prev_v, v], dim=-2)
        new_cache = (k, v)

        S = k.shape[-2]  # full sequence length after caching

        scale = self.head_dim ** 0.5
        attn = (q @ k.transpose(-2, -1)) / scale  # (B, n_heads, T, S)

        # Causal mask: when using cache, T=1 (new token), S=full length
        mask = self.causal_mask[:, :, S - T : S, :S]
        attn = attn.masked_fill(mask == 0, float("-inf"))
        attn = F.softmax(attn, dim=-1)

        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out), new_cache


class TransformerBlockCached(nn.Module):
    """Pre-norm transformer block with KV cache support."""

    def __init__(self, embed_dim, n_heads, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttentionWithCache(embed_dim, n_heads, max_seq_len)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
        )

    def forward(self, x, cache=None):
        attn_out, new_cache = self.attn(self.ln1(x), cache=cache)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x, new_cache


class CachedGPT2(nn.Module):
    """GPT-2 with KV cache support for efficient generation."""

    def __init__(self, vocab_size, embed_dim, n_heads, n_layers, max_seq_len):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(max_seq_len, embed_dim)
        self.blocks = nn.ModuleList(
            [TransformerBlockCached(embed_dim, n_heads, max_seq_len)
             for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)

    def forward(self, idx, caches=None, start_pos=0):
        B, T = idx.shape
        positions = torch.arange(start_pos, start_pos + T, device=idx.device).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(positions)

        new_caches = []
        for i, block in enumerate(self.blocks):
            layer_cache = caches[i] if caches is not None else None
            x, new_cache = block(x, cache=layer_cache)
            new_caches.append(new_cache)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits, new_caches


print("MultiHeadAttentionWithCache, TransformerBlockCached, CachedGPT2 defined.")

In [ ]:
# Copy weights from the trained model to the cached version
cached_model = CachedGPT2(VOCAB_SIZE, EMBED_DIM, N_HEADS, N_LAYERS, MAX_SEQ_LEN).to(device)

cached_state = cached_model.state_dict()
orig_state = model.state_dict()
for key in cached_state:
    if key in orig_state:
        cached_state[key] = orig_state[key]

cached_model.load_state_dict(cached_state)
cached_model.eval()
print("Weights copied from trained model to CachedGPT2.")


@torch.no_grad()
def generate_with_cache(model, prompt_tokens, max_new_tokens=50, temperature=1.0):
    """Generate using KV cache for efficiency."""
    model.eval()
    tokens = list(prompt_tokens)

    # Prefill: process the entire prompt at once
    x = torch.tensor([tokens], dtype=torch.long, device=device)
    logits, caches = model(x, caches=None, start_pos=0)
    next_logits = logits[:, -1, :]

    if temperature == 0:
        next_token = next_logits.argmax(dim=-1).item()
    else:
        probs = F.softmax(next_logits / temperature, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1).item()
    tokens.append(next_token)

    # Decode: process one token at a time, reusing the cache
    for step in range(1, max_new_tokens):
        x = torch.tensor([[next_token]], dtype=torch.long, device=device)
        start_pos = len(tokens) - 1
        logits, caches = model(x, caches=caches, start_pos=start_pos)
        next_logits = logits[:, -1, :]

        if temperature == 0:
            next_token = next_logits.argmax(dim=-1).item()
        else:
            probs = F.softmax(next_logits / temperature, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
        tokens.append(next_token)

    return tokens


print("generate_with_cache() defined.")

In [ ]:
# Verify that cached and non-cached generation produce the same greedy output
prompt_tokens = encode("the cat")

greedy_no_cache = generate(model, prompt_tokens, max_new_tokens=40, temperature=0)
greedy_with_cache = generate_with_cache(cached_model, prompt_tokens, max_new_tokens=40,
                                         temperature=0)

print("Without cache:", decode(greedy_no_cache))
print("With cache:   ", decode(greedy_with_cache))
print(f"\nOutputs match: {greedy_no_cache == greedy_with_cache}")

### Speed Comparison

Now let us measure the wall-clock time to generate sequences of different lengths,
with and without the KV cache.

In [ ]:
# Benchmark: generation time with and without KV cache
prompt_tokens = encode("the ")
gen_lengths = [20, 40, 60, 80, 100]
times_no_cache = []
times_with_cache = []

for n_tokens in gen_lengths:
    # Without cache
    t0 = time.time()
    for _ in range(3):  # average over 3 runs
        _ = generate(model, prompt_tokens, max_new_tokens=n_tokens, temperature=0)
    times_no_cache.append((time.time() - t0) / 3)

    # With cache
    t0 = time.time()
    for _ in range(3):
        _ = generate_with_cache(cached_model, prompt_tokens, max_new_tokens=n_tokens,
                                temperature=0)
    times_with_cache.append((time.time() - t0) / 3)

print(f"{'Tokens':<10} {'No Cache (s)':<15} {'With Cache (s)':<15} {'Speedup':<10}")
print("-" * 50)
for n, t_nc, t_wc in zip(gen_lengths, times_no_cache, times_with_cache):
    speedup = t_nc / t_wc if t_wc > 0 else float('inf')
    print(f"{n:<10} {t_nc:<15.4f} {t_wc:<15.4f} {speedup:<10.2f}x")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: absolute times
ax = axes[0]
ax.plot(gen_lengths, times_no_cache, "o-", label="Without KV cache", linewidth=2)
ax.plot(gen_lengths, times_with_cache, "s-", label="With KV cache", linewidth=2)
ax.set_xlabel("Number of generated tokens")
ax.set_ylabel("Time (seconds)")
ax.set_title("Generation Time: Cache vs No Cache")
ax.legend()
ax.grid(True, alpha=0.3)

# Right: speedup factor
ax = axes[1]
speedups = [t_nc / t_wc if t_wc > 0 else 0
            for t_nc, t_wc in zip(times_no_cache, times_with_cache)]
ax.bar(range(len(gen_lengths)), speedups, color="coral", alpha=0.8)
ax.set_xticks(range(len(gen_lengths)))
ax.set_xticklabels([str(n) for n in gen_lengths])
ax.set_xlabel("Number of generated tokens")
ax.set_ylabel("Speedup (x)")
ax.set_title("KV Cache Speedup Factor")
ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

The KV cache provides a consistent speedup that grows with sequence length. At our
tiny scale the absolute times are small, but the relative savings are significant.
In production models with thousands of tokens and billions of parameters, the KV cache
is essential -- without it, generation would be impractically slow.

---
## 9. Visualizations

### Entropy and Top-1 Probability vs Temperature

A closer look at how temperature reshapes the probability mass over the full vocabulary.

In [ ]:
temperatures_fine = np.linspace(0.1, 3.0, 50)
entropies = []
top1_probs = []

for T in temperatures_fine:
    scaled = raw_logits / T
    probs = F.softmax(scaled, dim=-1)
    log_probs = torch.log(probs + 1e-10)
    entropy = -(probs * log_probs).sum().item()
    entropies.append(entropy)
    top1_probs.append(probs.max().item())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(temperatures_fine, entropies, linewidth=2, color="steelblue")
axes[0].set_xlabel("Temperature")
axes[0].set_ylabel("Entropy (nats)")
axes[0].set_title("Distribution Entropy vs Temperature")
axes[0].axvline(x=1.0, color="red", linestyle="--", alpha=0.5, label="T=1.0")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(temperatures_fine, top1_probs, linewidth=2, color="coral")
axes[1].set_xlabel("Temperature")
axes[1].set_ylabel("Probability of top token")
axes[1].set_title("Top-1 Token Probability vs Temperature")
axes[1].axvline(x=1.0, color="red", linestyle="--", alpha=0.5, label="T=1.0")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

As temperature increases, entropy rises (the distribution becomes more uniform) and
the probability of the top token drops. The relationship is smooth and monotonic,
which is why temperature is such a convenient control knob.

### Top-k: Probability Mass Retained

In [ ]:
k_range = list(range(1, 80))
mass_retained = [sorted_probs[:k].sum() for k in k_range]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_range, mass_retained, linewidth=2, color="steelblue")
ax.axhline(y=0.9, color="red", linestyle="--", alpha=0.6, label="90% mass")
ax.axhline(y=0.95, color="orange", linestyle="--", alpha=0.6, label="95% mass")

k_90 = next(k for k in k_range if sorted_probs[:k].sum() >= 0.9)
k_95 = next(k for k in k_range if sorted_probs[:k].sum() >= 0.95)
ax.annotate(f"k={k_90}", xy=(k_90, 0.9), xytext=(k_90 + 5, 0.85),
            arrowprops=dict(arrowstyle="->", color="red"), fontsize=10, color="red")
ax.annotate(f"k={k_95}", xy=(k_95, 0.95), xytext=(k_95 + 5, 0.88),
            arrowprops=dict(arrowstyle="->", color="orange"), fontsize=10, color="orange")

ax.set_xlabel("k (number of top tokens kept)")
ax.set_ylabel("Probability mass retained")
ax.set_title("Top-k: Probability Mass Retained vs k")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Ablation: Comparing Sampling Strategies

Let us generate from the same prompt using different strategies and compare the outputs
side by side.

### Diversity Measurement

We can quantify the diversity of generated text by counting the number of unique
n-grams in the output. More diverse strategies produce more unique n-grams.

In [ ]:
prompt = "the cat sat on"
prompt_tokens = encode(prompt)
n_gen = 60

strategies = {
    "Greedy":              dict(temperature=0),
    "Temp=0.5":            dict(temperature=0.5),
    "Temp=0.8":            dict(temperature=0.8),
    "Temp=1.0":            dict(temperature=1.0),
    "Top-k=5":             dict(temperature=1.0, top_k=5),
    "Top-k=10":            dict(temperature=1.0, top_k=10),
    "Top-p=0.8":           dict(temperature=1.0, top_p=0.8),
    "Top-p=0.9":           dict(temperature=1.0, top_p=0.9),
    "Top-k=10 + Temp=0.8": dict(temperature=0.8, top_k=10),
    "Top-p=0.9 + Temp=0.8":dict(temperature=0.8, top_p=0.9),
}

print(f"Prompt: '{prompt}'")
print("=" * 70)
for name, kwargs in strategies.items():
    tokens = generate(model, prompt_tokens, max_new_tokens=n_gen, **kwargs)
    text = decode(tokens)
    print(f"  [{name:>22}]  {text}")
    print()

In [ ]:
def count_unique_ngrams(token_list, n=3):
    """Count the number of unique n-grams in a token sequence."""
    ngrams = set()
    for i in range(len(token_list) - n + 1):
        ngrams.add(tuple(token_list[i:i + n]))
    return len(ngrams)


n_samples = 5
strategy_names = []
diversity_scores = []

for name, kwargs in strategies.items():
    unique_counts = []
    for _ in range(n_samples):
        tokens = generate(model, prompt_tokens, max_new_tokens=80, **kwargs)
        generated_part = tokens[len(prompt_tokens):]
        unique_counts.append(count_unique_ngrams(generated_part, n=3))
    strategy_names.append(name)
    diversity_scores.append(np.mean(unique_counts))

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(range(len(strategy_names)), diversity_scores, color="steelblue", alpha=0.8)
ax.set_yticks(range(len(strategy_names)))
ax.set_yticklabels(strategy_names)
ax.set_xlabel("Mean unique 3-grams (over 5 samples)")
ax.set_title("Output Diversity by Sampling Strategy")
ax.grid(True, alpha=0.3, axis="x")

for bar, val in zip(bars, diversity_scores):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

Greedy decoding produces the fewest unique n-grams (most repetitive). Higher temperatures
and larger top-k / top-p values increase diversity. The combined strategies (e.g.,
top-k + lower temperature) provide a middle ground.

### KV Cache Memory Usage

The cache stores K and V tensors for every layer. Let us measure how the cache size
grows with sequence length.

In [ ]:
def measure_cache_size(caches):
    """Compute total memory used by KV caches in bytes."""
    total = 0
    for k, v in caches:
        total += k.nelement() * k.element_size()
        total += v.nelement() * v.element_size()
    return total


prompt_tokens = encode("the ")
seq_lengths = []
cache_sizes_bytes = []

cached_model.eval()
tokens = list(prompt_tokens)

with torch.no_grad():
    # Prefill
    x = torch.tensor([tokens], dtype=torch.long, device=device)
    logits, caches = cached_model(x, caches=None, start_pos=0)
    next_token = logits[:, -1, :].argmax(dim=-1).item()
    tokens.append(next_token)
    seq_lengths.append(len(tokens))
    cache_sizes_bytes.append(measure_cache_size(caches))

    # Generate and track cache growth
    for step in range(80):
        x = torch.tensor([[next_token]], dtype=torch.long, device=device)
        start_pos = len(tokens) - 1
        logits, caches = cached_model(x, caches=caches, start_pos=start_pos)
        next_token = logits[:, -1, :].argmax(dim=-1).item()
        tokens.append(next_token)
        seq_lengths.append(len(tokens))
        cache_sizes_bytes.append(measure_cache_size(caches))

cache_sizes_kb = [b / 1024 for b in cache_sizes_bytes]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(seq_lengths, cache_sizes_kb, linewidth=2, color="coral")
ax.set_xlabel("Sequence length (tokens)")
ax.set_ylabel("KV cache size (KB)")
ax.set_title("KV Cache Memory Growth During Generation")
ax.grid(True, alpha=0.3)
ax.annotate("Linear growth: O(n_layers * seq_len * embed_dim)",
            xy=(seq_lengths[-1] // 2, cache_sizes_kb[-1] // 2),
            fontsize=10, color="gray", style="italic")
plt.tight_layout()
plt.show()

print(f"\nCache size at sequence length {seq_lengths[-1]}: {cache_sizes_kb[-1]:.2f} KB")
print(f"Per-token cache cost: {(cache_sizes_bytes[-1] / seq_lengths[-1]):.1f} bytes")
print(f"Breakdown: {N_LAYERS} layers x 2 (K+V) x {N_HEADS} heads x "
      f"{EMBED_DIM // N_HEADS} head_dim x 4 bytes = "
      f"{N_LAYERS * 2 * N_HEADS * (EMBED_DIM // N_HEADS) * 4} bytes per token")

The cache grows linearly with sequence length. Each new token adds a fixed amount of
memory (determined by `n_layers * 2 * embed_dim * element_size`). For our tiny model
the numbers are small, but for production LLMs with embed_dim=4096+ and 32+ layers,
the KV cache can consume gigabytes of GPU memory -- which is why techniques like
grouped-query attention (GQA), sliding-window attention, and cache eviction strategies
are critical in practice.

---
## 11. Putting It All Together

In practice, generation APIs expose all these knobs simultaneously. Here is a final
demonstration combining temperature, top-k, and top-p in a single call.

In [ ]:
configs = {
    "Conservative (T=0.3, top-k=10, top-p=0.9)":  dict(temperature=0.3, top_k=10, top_p=0.9),
    "Balanced (T=0.7, top-k=40, top-p=0.95)":     dict(temperature=0.7, top_k=40, top_p=0.95),
    "Creative (T=1.0, top-k=50, top-p=0.95)":     dict(temperature=1.0, top_k=50, top_p=0.95),
    "Wild (T=1.2, top-p=0.98)":                   dict(temperature=1.2, top_p=0.98),
}

prompt = "the cat"
prompt_tokens = encode(prompt)

print(f"Prompt: '{prompt}'")
print("=" * 70)
for name, kwargs in configs.items():
    print(f"\n  [{name}]")
    for i in range(3):
        tokens = generate(model, prompt_tokens, max_new_tokens=60, **kwargs)
        print(f"    Sample {i+1}: {decode(tokens)}")

---
## Key Insights

1. **Autoregressive generation is inherently sequential.** Each new token depends on
   all previous tokens, making the generation loop the primary bottleneck. The KV cache
   mitigates this by avoiding redundant computation, but the loop itself cannot be
   parallelized.

2. **Greedy decoding is deterministic but degenerate.** Always picking argmax leads to
   repetitive, low-diversity text. Real applications almost never use pure greedy decoding
   for open-ended generation (though it remains useful for tasks with a single correct
   answer, like translation or summarization).

3. **Temperature controls the entropy-quality trade-off.** Low temperature yields
   confident but repetitive text; high temperature yields diverse but often incoherent
   text. There is no universally correct temperature -- it depends on the task.

4. **Top-k and top-p truncate the tail.** Both prevent sampling very unlikely tokens,
   but they differ in adaptivity. Top-k uses a fixed candidate set size; top-p adapts
   to the shape of the distribution. In practice, top-p (nucleus sampling) is more
   commonly used because it naturally handles both peaked and flat distributions.

5. **Top-k and top-p are complementary to temperature.** Temperature reshapes the entire
   distribution; top-k/top-p truncate it. Combining them (e.g., temperature=0.8 with
   top-p=0.95) is standard practice and gives the best results.

6. **The KV cache trades memory for speed.** Cache memory grows linearly with sequence
   length, but the speedup grows superlinearly (since it eliminates the quadratic
   recomputation). For long sequences and large models, managing the KV cache is one of
   the central challenges in serving LLMs efficiently.

7. **Cache size is a real constraint.** For a model with `d_model=4096` and `n_layers=32`
   using float16, each token adds `32 * 2 * 4096 * 2 = 512 KB` to the cache. A 4096-token
   context would require ~2 GB of cache per sequence -- which is why batch serving of
   long-context LLMs demands careful memory management (paged attention, cache eviction,
   quantized KV, etc.).